In [48]:
!pip install scikit-learn numpy nltk -q

In [49]:
import numpy as np
from itertools import combinations
from collections import defaultdict
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import nltk
from nltk.corpus import words as nltk_words
import tensorflow_datasets as tfds
from torchvision.datasets import EMNIST
import torchvision.transforms as transforms
import warnings
warnings.filterwarnings("ignore")

## Factor algebra

In [50]:
def make_factor(var, card, val=None):
    """Create a factor dict with variables, cardinalities, and values."""
    size = int(np.prod(card))
    return {
        "var":  list(var),
        "card": list(card),
        "val":  np.ones(size) if val is None else np.array(val, dtype=float),
    }


def idx_to_assign(idx, cards):
    assign = []
    for c in reversed(cards):
        assign.append(idx % c)
        idx //= c
    return list(reversed(assign))


def assign_to_idx(assign, cards):
    idx, stride = 0, 1
    for i in range(len(assign) - 1, -1, -1):
        idx += assign[i] * stride
        stride *= cards[i]
    return idx


def factor_product(A, B):
    if not A["var"]: return B
    if not B["var"]: return A
    all_vars  = list(dict.fromkeys(A["var"] + B["var"]))
    card_map  = {v: c for v, c in zip(A["var"], A["card"])}
    card_map.update({v: c for v, c in zip(B["var"], B["card"])})
    all_cards = [card_map[v] for v in all_vars]
    C = make_factor(all_vars, all_cards)
    for i in range(len(C["val"])):
        asgn = idx_to_assign(i, all_cards)
        amap = {v: asgn[j] for j, v in enumerate(all_vars)}
        ia = assign_to_idx([amap[v] for v in A["var"]], A["card"])
        ib = assign_to_idx([amap[v] for v in B["var"]], B["card"])
        C["val"][i] = A["val"][ia] * B["val"][ib]
    return C


def factor_marginalize(F, var):
    if var not in F["var"]: return F
    idx       = F["var"].index(var)
    new_vars  = [v for v in F["var"]  if v != var]
    new_cards = [c for i, c in enumerate(F["card"]) if i != idx]
    G = make_factor(new_vars, new_cards if new_cards else [1])
    for i in range(len(F["val"])):
        asgn     = idx_to_assign(i, F["card"])
        new_asgn = [asgn[j] for j in range(len(asgn)) if j != idx]
        gi = assign_to_idx(new_asgn, new_cards) if new_cards else 0
        G["val"][gi] += F["val"][i]
    return G


def normalize_factor(F):
    s = F["val"].sum()
    G = {"var": F["var"], "card": F["card"], "val": F["val"] / s if s > 0 else F["val"].copy()}
    return G



## Data loading and preprocessing

In [51]:
TARGET_H, TARGET_W = 16, 8   # Match original PA3 image dimensions
K = 26                        # 26-letter alphabet (a–z)

In [52]:
def load_emnist(n_train=20000, n_test=4000):
    print("Loading EMNIST Letters...")
    try:
        transform = transforms.Compose([
            transforms.ToTensor(),
        ])

        train_ds = EMNIST(root="/tmp/emnist", split="letters", train=True,
                          download=True, transform=transform)
        test_ds  = EMNIST(root="/tmp/emnist", split="letters", train=False,
                          download=True, transform=transform)

        def ds_to_numpy(ds, n):
            imgs, labels = [], []
            for i in range(min(n, len(ds))):
                img, label = ds[i]
                imgs.append(img.numpy().squeeze())   # (28, 28)
                labels.append(int(label) - 1)        # 1-26 → 0-25
            return np.array(imgs), np.array(labels)

        X_tr_img, y_tr = ds_to_numpy(train_ds, n_train)
        X_te_img, y_te = ds_to_numpy(test_ds,  n_test)

    except ImportError:
        ds = tfds.load("emnist/letters", split=["train", "test"], as_supervised=True)

        def tfds_to_numpy(split_ds, n):
            imgs, labels = [], []
            for img, label in split_ds.take(n):
                imgs.append(img.numpy().squeeze())
                labels.append(int(label) - 1)
            return np.array(imgs), np.array(labels)

        X_tr_img, y_tr = tfds_to_numpy(ds[0], n_train)
        X_te_img, y_te = tfds_to_numpy(ds[1], n_test)

    X_tr = _resize_and_flatten(X_tr_img)
    X_te = _resize_and_flatten(X_te_img)

    print(f"  Train: {X_tr.shape}, Test: {X_te.shape}")
    print(f"  Classes: {len(np.unique(y_tr))}, Labels: {y_tr.min()}–{y_tr.max()}")
    return X_tr, X_te, y_tr, y_te


def _resize_and_flatten(imgs):
    """imgs: (N, 28, 28) → (N, 128) binary flattened."""
    out = []
    for img in imgs:
      # EMNIST Letters are rotated + flipped compared to the usual reading direction
      # Need to transpose to get the image in the correct orientation before resizing
        img = np.rot90(img, k=3)   # rotate 270 độ
        img = np.fliplr(img)        # flip ngang

        h_idx = np.linspace(0, 27, 16).astype(int)
        w_idx = np.linspace(0, 27,  8).astype(int)
        small  = img[np.ix_(h_idx, w_idx)]
        binary = (small > 0.5).astype(np.float32)
        out.append(binary.flatten())
    return np.array(out)

## Image model (singleton factors)

In [53]:
class CharCNN(nn.Module):
    """
    Small CNN for single character image classification.
    Input: (batch, 1, 16, 8) binary images.
    Output: (batch, K) log-probabilities.
    """
    def __init__(self, K=26):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),  # (B, 32, 16, 8)
            nn.ReLU(),
            nn.MaxPool2d(2),                              # (B, 32, 8, 4)
            nn.Conv2d(32, 64, kernel_size=3, padding=1), # (B, 64, 8, 4)
            nn.ReLU(),
            nn.MaxPool2d(2),                              # (B, 64, 4, 2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),          # (B, 64*4*2) = (B, 512)
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, K),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


In [54]:
class ImageModel:
    def __init__(self, K=26, epochs=15, batch_size=256, lr=1e-3):
        self.K          = K
        self.epochs     = epochs
        self.batch_size = batch_size
        self.lr         = lr
        self.device     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.net        = CharCNN(K).to(self.device)

    def fit(self, X, y):
        """
        X: (N, 128) flattened binary float32 arrays
        y: (N,)     integer labels 0..K-1
        """
        # Reshape to (N, 1, 16, 8) for Conv2d
        X_t = torch.tensor(X, dtype=torch.float32).reshape(-1, 1, 16, 8)
        y_t = torch.tensor(y, dtype=torch.long)

        loader = DataLoader(
            TensorDataset(X_t, y_t),
            batch_size=self.batch_size, shuffle=True
        )
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(self.net.parameters(), lr=self.lr)

        self.net.train()
        for epoch in range(self.epochs):
            total_loss = 0
            for xb, yb in loader:
                xb, yb = xb.to(self.device), yb.to(self.device)
                optimizer.zero_grad()
                loss = criterion(self.net(xb), yb)
                loss.backward()
                optimizer.step()
                total_loss += loss.item()
            print(f"  Epoch {epoch+1}/{self.epochs}  loss={total_loss/len(loader):.4f}")
        return self

    def compute_image_factor(self, img_flat):
        """
        Return P(k | img) as a numpy array of length K.
        img_flat: 1-D array độ dài 128.
        """
        self.net.eval()
        with torch.no_grad():
            x = torch.tensor(img_flat, dtype=torch.float32)
            x = x.reshape(1, 1, 16, 8).to(self.device)
            logits = self.net(x)                    # (1, K)
            probs  = torch.softmax(logits, dim=1)   # (1, K)
        return probs.cpu().numpy().flatten()        # (K,)

    def score(self, X, y):
        self.net.eval()
        X_t = torch.tensor(X, dtype=torch.float32).reshape(-1, 1, 16, 8).to(self.device)
        with torch.no_grad():
            preds = self.net(X_t).argmax(dim=1).cpu().numpy()
        return (preds == y).mean()

## Pairwise and triplet models from a corpus

In [55]:

def build_pairwise_model(word_list, K=26, smoothing=1.0):
    """
    Estimate P(c_j | c_i) from a word corpus using Laplace smoothing.
    Returns a (K, K) matrix where entry [i,j] = factor value for (i, j) bigram.

    word_list: list of lowercase strings, e.g. ["the", "cat", ...]
    """
    counts = np.ones((K, K)) * smoothing
    for word in word_list:
        word = word.lower()
        for a, b in zip(word[:-1], word[1:]):
            ai, bi = ord(a) - ord('a'), ord(b) - ord('a')
            if 0 <= ai < K and 0 <= bi < K:
                counts[ai, bi] += 1
    # Normalise rows → conditional distribution, but keep as factor values (not log)
    row_sums = counts.sum(axis=1, keepdims=True)
    return counts / row_sums


def build_triplet_model(word_list, K=26, smoothing=1e-3, top_n=500):
    """
    Collect the top_n most common character triplets from a corpus.
    Returns a list of dicts: [{"chars": (a,b,c), "val": float}, ...]

    Only triplets with factor value > 1 (i.e., more frequent than baseline)
    are kept, which is what the original assignment does.
    """
    counts = defaultdict(float)
    total  = 0
    for word in word_list:
        word = word.lower()
        for a, b, c in zip(word[:-2], word[1:-1], word[2:]):
            ai = ord(a) - ord('a')
            bi = ord(b) - ord('a')
            ci = ord(c) - ord('a')
            if all(0 <= x < K for x in [ai, bi, ci]):
                counts[(ai, bi, ci)] += 1
                total += 1

    baseline = total / (K ** 3)
    triplets = []
    for (a, b, c), cnt in sorted(counts.items(), key=lambda x: -x[1])[:top_n]:
        val = (cnt + smoothing) / (baseline + smoothing)
        if val > 1.0:
            triplets.append({"chars": (a, b, c), "val": val})
    return triplets


def load_english_words():
    """
    Return a list of common English words for corpus estimation.
    Uses the NLTK words corpus if available; otherwise falls back to a
    small built-in list sufficient for demonstration.
    """
    try:
        nltk.download("words", quiet=True)
        return [w.lower() for w in nltk_words.words() if w.isalpha()]
    except Exception:
        # Minimal fallback — replace with a larger list for real use
        return [
            "the", "and", "for", "are", "but", "not", "you", "all", "can",
            "her", "was", "one", "our", "out", "day", "get", "has", "him",
            "his", "how", "man", "new", "now", "old", "see", "two", "way",
            "who", "boy", "did", "its", "let", "put", "say", "she", "too",
            "use", "cat", "dog", "hat", "run", "sit", "big", "red", "hot",
            "cold", "time", "year", "when", "come", "here", "just", "know",
            "take", "into", "that", "this", "with", "have", "from", "they",
            "will", "been", "good", "much", "some", "what", "well", "also",
            "back", "after", "first", "never", "these", "think", "where",
            "being", "every", "great", "might", "still", "those", "three",
            "while", "world", "about", "before", "could", "would", "should",
        ]



## Factor construction

In [56]:
def compute_singleton_factors(images_flat, image_model):
    """
    Build one singleton factor per character image.

    images_flat: list of 1-D arrays, one per character position.
    Returns: list of factor dicts, factor[i] covers variable i.
    """
    factors = []
    for i, img in enumerate(images_flat):
        probs = image_model.compute_image_factor(img)
        factors.append(make_factor([i], [image_model.K], probs))
    return factors


def compute_pairwise_factors(n_chars, pairwise_model, K):
    """
    Build (n_chars - 1) pairwise factors from the transition matrix.

    pairwise_model: (K, K) numpy array where [i,j] = factor value for bigram (i,j).
    """
    if n_chars < 2:
        return []
    factors = []
    for i in range(n_chars - 1):
        val = pairwise_model.flatten(order='C')  # row-major: val[i*K+j] = pairwise[i,j]
        factors.append(make_factor([i, i + 1], [K, K], val))
    return factors


def compute_triplet_factors(n_chars, triplet_list, K):
    """
    Build (n_chars - 2) triplet factors.

    triplet_list: output of build_triplet_model(), list of {"chars": (a,b,c), "val": float}.
    The val table defaults to 1 for all assignments; entries in triplet_list override.
    """
    if n_chars < 3:
        return []
    val_table = np.ones(K ** 3)
    for t in triplet_list:
        a, b, c = t["chars"]
        idx = assign_to_idx([a, b, c], [K, K, K])
        val_table[idx] = t["val"]
    factors = []
    for i in range(n_chars - 2):
        factors.append(make_factor([i, i + 1, i + 2], [K, K, K], val_table.copy()))
    return factors


def image_similarity(img1, img2):
    """
    Cosine-distance-based similarity score between two binary images.
    Mirrors ImageSimilarity.m from the original assignment exactly.
    """
    a, b = img1.flatten().astype(float), img2.flatten().astype(float)
    norm_a, norm_b = np.linalg.norm(a), np.linalg.norm(b)
    if norm_a == 0 or norm_b == 0:
        return 1.0
    mean_sim = 0.283
    cos_dist = (a @ b) / (norm_a * norm_b)
    diff = (cos_dist - mean_sim) ** 2
    if cos_dist > mean_sim:
        return 1.0 + 5 * diff
    else:
        return 1.0 / (1.0 + 5 * diff)


def compute_similarity_factor(images_flat, K, i, j):
    """
    Factor between positions i and j encoding visual similarity.
    Value = sim if C_i == C_j, else 1.
    """
    sim = image_similarity(images_flat[i], images_flat[j])
    val = np.ones((K, K))
    np.fill_diagonal(val, sim)
    return make_factor([i, j], [K, K], val.flatten())


def compute_all_similarity_factors(images_flat, K):
    """Compute C(n,2) similarity factors for all pairs of character images."""
    n = len(images_flat)
    return [compute_similarity_factor(images_flat, K, i, j)
            for i, j in combinations(range(n), 2)]


def choose_top_similarity_factors(all_factors, F):
    """Keep the F similarity factors with the highest similarity score."""
    if len(all_factors) <= F:
        return all_factors
    scores = [f["val"][0] for f in all_factors]  # diagonal entry = sim score
    idx    = np.argsort(scores)[::-1][:F]
    return [all_factors[i] for i in idx]


def build_ocr_network(images_flat, image_model, pairwise_model=None,
                      triplet_list=None, use_similarity=True, top_sim=2):
    """
    Assemble the full Markov network for one word.

    images_flat: list of 1-D numpy arrays (one per character).
    Returns: list of all factor dicts.
    """
    n = len(images_flat)
    K = image_model.K

    all_factors = compute_singleton_factors(images_flat, image_model)

    if pairwise_model is not None:
        all_factors += compute_pairwise_factors(n, pairwise_model, K)

    if triplet_list is not None:
        all_factors += compute_triplet_factors(n, triplet_list, K)

    if use_similarity:
        sim_all = compute_all_similarity_factors(images_flat, K)
        all_factors += choose_top_similarity_factors(sim_all, top_sim)

    return all_factors

## Loopy Belief Propagation

In [57]:

def run_loopy_bp(factors, n_vars, K, max_iter=30, tol=1e-6):
    """
    Run Loopy Belief Propagation and return the MAP assignment.

    Parameters
    ----------
    factors  : list of factor dicts (output of build_ocr_network)
    n_vars   : number of variable nodes (= word length)
    K        : alphabet size
    max_iter : maximum number of BP iterations
    tol      : convergence threshold on max message change

    Returns
    -------
    pred : list of length n_vars, each entry in 0..K-1
    """
    # Index factors by their scope for efficient message computation
    # Messages: msg[(factor_idx, var)] = numpy array of length K

    # Assign each factor a unique index
    fac_list = factors

    # Initialise all messages to uniform
    messages = {}
    for fi, f in enumerate(fac_list):
        for v in f["var"]:
            messages[(fi, v)] = np.ones(K) / K

    for iteration in range(max_iter):
        old_messages = {k: v.copy() for k, v in messages.items()}

        # Variable factor-> factor messages (product of incoming factorfactor->var messages
        # from all OTHER factors involving this variable)
        var_to_fac = {}
        for fi, f in enumerate(fac_list):
            for v in f["var"]:
                # Incoming from all factors except fi
                incoming = np.ones(K)
                for fj, g in enumerate(fac_list):
                    if fj != fi and v in g["var"]:
                        incoming *= messages[(fj, v)]
                incoming /= incoming.sum() + 1e-300
                var_to_fac[(v, fi)] = incoming

        # Factor factor-> variable messages
        for fi, f in enumerate(fac_list):
            f_vars  = f["var"]
            f_cards = f["card"]
            f_val   = f["val"].reshape(f_cards)

            for target_v in f_vars:
                target_idx = f_vars.index(target_v)
                # Multiply factor by incoming varfactor->fac messages for all OTHER vars
                msg = f_val.copy().astype(float)
                for other_v in f_vars:
                    if other_v == target_v:
                        continue
                    other_idx = f_vars.index(other_v)
                    inc = var_to_fac[(other_v, fi)]
                    # Broadcast inc along other_idx axis
                    shape = [1] * len(f_vars)
                    shape[other_idx] = K
                    msg *= inc.reshape(shape)

                # Marginalise out all axes except target_idx
                axes = tuple(i for i in range(len(f_vars)) if i != target_idx)
                if axes:
                    msg = msg.sum(axis=axes)
                msg = msg.flatten()
                msg /= msg.sum() + 1e-300
                messages[(fi, target_v)] = msg

        # Check convergence
        delta = max(
            np.max(np.abs(messages[k] - old_messages[k]))
            for k in messages
        )
        if delta < tol:
            break

    # Compute beliefs: product of all factorfactor->var messages for each variable
    beliefs = []
    for v in range(n_vars):
        b = np.ones(K)
        for fi, f in enumerate(fac_list):
            if v in f["var"]:
                b *= messages[(fi, v)]
        b /= b.sum() + 1e-300
        beliefs.append(b)

    pred = [int(np.argmax(b)) for b in beliefs]
    return pred



## Evaluation



In [58]:

def score_predictions(words_true, words_pred):
    """
    Compute character-level and word-level accuracy.

    words_true, words_pred: lists of lists of int (label indices).
    Returns: (char_acc, word_acc) as floats in [0, 1].
    """
    n_chars_correct = 0
    n_chars_total   = 0
    n_words_correct = 0

    for true_word, pred_word in zip(words_true, words_pred):
        correct = [t == p for t, p in zip(true_word, pred_word)]
        n_chars_correct += sum(correct)
        n_chars_total   += len(true_word)
        if all(correct):
            n_words_correct += 1

    char_acc = n_chars_correct / n_chars_total if n_chars_total > 0 else 0.0
    word_acc = n_words_correct / len(words_true) if words_true else 0.0
    return char_acc, word_acc


def evaluate_model(test_words, image_model, pairwise_model=None,
                   triplet_list=None, use_similarity=True, top_sim=2,
                   max_bp_iter=30):
    """
    Run the full pipeline on a list of test words and return accuracy.

    test_words: list of dicts {"images": [flat_array, ...], "labels": [int, ...]}
    """
    words_true, words_pred = [], []
    for word in test_words:
        images_flat = word["images"]
        n           = len(images_flat)
        K           = image_model.K
        factors     = build_ocr_network(
            images_flat, image_model, pairwise_model,
            triplet_list, use_similarity, top_sim
        )
        pred = run_loopy_bp(factors, n, K, max_iter=max_bp_iter)
        words_true.append(word["labels"])
        words_pred.append(pred)
    return score_predictions(words_true, words_pred)


## Synthetic word generation from EMNIST

In [59]:
def make_synthetic_words(X, y, word_list, n_words=200, rng_seed=0):
    rng = np.random.default_rng(rng_seed)
    by_class = defaultdict(list)
    for xi, yi in zip(X, y):
        by_class[yi].append(xi)

    candidates = [w.lower() for w in word_list
                  if 2 <= len(w) <= 8 and w.isalpha()]

    candidates = list(rng.choice(candidates, size=min(len(candidates), 2000), replace=False))

    words_out = []
    for word in candidates:
        labels = [ord(c) - ord('a') for c in word]
        if any(len(by_class[l]) == 0 for l in labels):
            continue
        images = [rng.choice(by_class[l]) for l in labels]
        words_out.append({"images": images, "labels": labels, "word": word})
        if len(words_out) >= n_words:
            break

    return words_out

## Ablation — compare factor configurations

In [60]:
def run_ablation(test_words, image_model, pairwise_model, triplet_list):
    """
    Compare four progressively richer factor configurations:
      (1) Singleton only
      (2) Singleton + pairwise
      (3) Singleton + pairwise + triplet
      (4) Singleton + pairwise + triplet + similarity

    Prints a results table and returns a dict of (char_acc, word_acc) per config.
    """
    configs = [
        ("Singleton only",            dict(pairwise_model=None,          triplet_list=None,         use_similarity=False)),
        ("+ Pairwise",                dict(pairwise_model=pairwise_model, triplet_list=None,         use_similarity=False)),
        ("+ Pairwise + Triplet",      dict(pairwise_model=pairwise_model, triplet_list=triplet_list, use_similarity=False)),
        ("+ Pairwise + Triplet + Sim",dict(pairwise_model=pairwise_model, triplet_list=triplet_list, use_similarity=True )),
    ]

    print(f"\n{'Config':<35} {'Char Acc':>10} {'Word Acc':>10}")
    print("-" * 57)

    results = {}
    for name, kwargs in configs:
        char_acc, word_acc = evaluate_model(test_words, image_model, **kwargs)
        print(f"{name:<35} {char_acc:>10.3f} {word_acc:>10.3f}")
        results[name] = (char_acc, word_acc)

    return results


In [61]:
def show_sample_predictions(test_words, image_model, pairwise_model,
                            triplet_list, n_show=10):
    """Print the root word and the predicted word to check by eye."""
    print(f"\n{'Ground truth':<20} {'Singleton only':<20} {'+ Pairwise':<20}")
    print("-" * 62)

    CHARS = [chr(ord('a') + i) for i in range(26)]

    for word_dict in test_words[:n_show]:
        images_flat = word_dict["images"]
        n           = len(images_flat)
        true_str    = word_dict.get("word", "".join(CHARS[l] for l in word_dict["labels"]))

        # Singleton only
        f1   = build_ocr_network(images_flat, image_model,
                                 pairwise_model=None, triplet_list=None,
                                 use_similarity=False)
        p1   = run_loopy_bp(f1, n, image_model.K)
        s1   = "".join(CHARS[i] for i in p1)

        # + Pairwise
        f2   = build_ocr_network(images_flat, image_model,
                                 pairwise_model=pairwise_model,
                                 triplet_list=None, use_similarity=False)
        p2   = run_loopy_bp(f2, n, image_model.K)
        s2   = "".join(CHARS[i] for i in p2)

        print(f"{true_str:<20} {s1:<20} {s2:<20}")

## Main entry point

In [67]:
def main():
    # 1. Load data
    X_tr, X_te, y_tr, y_te = load_emnist(n_train=20000, n_test=4000)

    # 2. Train image model
    print("Training image model...")
    image_model = ImageModel(K=K)
    image_model.fit(X_tr, y_tr)
    print(f"  Character classification accuracy: {image_model.score(X_te, y_te):.3f}")

    # 3. Build pairwise and triplet models from English corpus
    print("Building language models from corpus...")
    english_words = load_english_words()
    print(f"  Corpus size: {len(english_words)} words")
    pairwise_model = build_pairwise_model(english_words, K=K)
    triplet_list   = build_triplet_model(english_words, K=K, top_n=500)
    print(f"  Triplet factors: {len(triplet_list)} entries")

    # 4. Build synthetic test words
    print("Generating synthetic test words...")
    test_words = make_synthetic_words(X_te, y_te, english_words, n_words=200)
    print(f"  Words generated: {len(test_words)}")


    # 5. Ablation study
    show_sample_predictions(test_words, image_model, pairwise_model, triplet_list)
    print("\nRunning ablation study...")
    results = run_ablation(test_words, image_model, pairwise_model, triplet_list)

    return results


In [68]:
main()

Loading EMNIST Letters...
  Train: (20000, 128), Test: (4000, 128)
  Classes: 26, Labels: 0–25
Training image model...
  Epoch 1/15  loss=2.7118
  Epoch 2/15  loss=1.7076
  Epoch 3/15  loss=1.3891
  Epoch 4/15  loss=1.2314
  Epoch 5/15  loss=1.1293
  Epoch 6/15  loss=1.0537
  Epoch 7/15  loss=0.9911
  Epoch 8/15  loss=0.9498
  Epoch 9/15  loss=0.9032
  Epoch 10/15  loss=0.8731
  Epoch 11/15  loss=0.8411
  Epoch 12/15  loss=0.8182
  Epoch 13/15  loss=0.8079
  Epoch 14/15  loss=0.7807
  Epoch 15/15  loss=0.7561
  Character classification accuracy: 0.791
Building language models from corpus...
  Corpus size: 236734 words
  Triplet factors: 500 entries
Generating synthetic test words...
  Words generated: 4

Ground truth         Singleton only       + Pairwise          
--------------------------------------------------------------
abaca                abaca                abaca               
bead                 bead                 bead                
bed                  bed          

{'Singleton only': (1.0, 1.0),
 '+ Pairwise': (1.0, 1.0),
 '+ Pairwise + Triplet': (1.0, 1.0),
 '+ Pairwise + Triplet + Sim': (1.0, 1.0)}